# ACP Températures

## Les données

On dispose de températures mensuelles moyennes pour 15 villes françaises. On dispose aussi de leur position en latitude et longitude.

## Les bibliothèques

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb

sb.set_style("white")

from sklearn.decomposition import PCA
from sklearn.preprocessing import scale


# pour avoir les graphiques dans le notebook
%matplotlib inline 

## Lecture des données

In [ ]:
temp = pd.read_csv("data_temp.csv", sep=";", header=0)
temp.head(15)

Ajout de 2 colonnes `moy` et `amp` respectivement égales à la moyenne annuelles des température et à l’amplitude annuelle de chaque ville.

In [ ]:
temp['moy']=temp[temp.columns[1:13]].mean(axis=1) # axis=1 pour travailler sur les lignes
temp['amp']=temp[temp.columns[1:13]].max(axis=1)-temp[temp.columns[1:13]].min(axis=1)
temp


# Résumé et visualisation des donnees

Affichage d'un résumé statistique des toutes les variables sauf la longitude et la latitude

In [ ]:
tempn=temp[temp.columns[1:13]]
tempn.describe().round(3)

Diagramme en boîte à moustache des données sauf les variables latitude, longitude, moyenne et amplitude.

In [ ]:
plt.figure(figsize = (16, 8))
sb.boxplot(data = tempn)
plt.show()

Nuages de points 2 à 2 des données sauf les variables latitude, longitude, moyenne et amplitude.

In [ ]:
sb.pairplot(data = tempn)

Matrices des corrélation de toutes les variable sauf latitude, longitude, moyenne et amplitude.

In [ ]:
tempn.corr().round(3)

# Graphiques profil de température

Profil des température des villes de Brest, Lille et Montpellier.

In [ ]:
data_profil=temp.query('Ville=="Brest" | Ville=="Lille"| Ville=="Montpellier"')

data_profil=data_profil.melt(id_vars='Ville',
                             value_vars=tempn.columns,var_name='Mois', 
                             value_name='Températures' )
sb.lineplot(data=data_profil,
            x='Mois',y="Températures", 
            hue='Ville', style='Ville',
            markers=True, dashes=False)


# ACP

ACP centrée réduite des données de températures. Les 4 dernières variables seront misent comme variables quantitatives supplémentaires.

In [ ]:
n=temp.shape[0]
tempn_cr=scale(tempn)
t_pca=PCA()
t_pca.fit(tempn_cr)

In [ ]:
# correction dans les valeurs propres car variances d'échantillonage utilisée
eig = pd.DataFrame({
    "Dimension": ["Dim"+str(i+1) for i in range(12)],
    "Valeurs propres": t_pca.explained_variance_*(n-1)/n,
    "% expliquée": t_pca.explained_variance_ratio_* 100,
    "% expliquée cumulée": np.cumsum(t_pca.explained_variance_ratio_) * 100
})
eig.round(2)

In [ ]:
plt.figure(figsize = (16, 10))
sb.barplot(x = "Dimension", y = "% expliquée", data = eig, color = "steelblue")
plt.show()

Coordonnées des individus

In [ ]:
coord_ind = t_pca.transform(tempn_cr)
coord_ind_df = pd.DataFrame({
    "Dim1" : coord_ind[:,0], 
    "Dim2" : coord_ind[:,1], 
})
coord_ind_df.index=temp.Ville

# Résultat 
coord_ind_df.round(3)

Contribution individus

In [ ]:
# contribution individus poids (n-1) sur les individus au lieu de n
contrib_ind_df=np.square(coord_ind_df).div((n-1)*t_pca.explained_variance_[:2], axis='columns')*100
contrib_ind_df.round(3)

Graphique des contributions des individus

In [ ]:
plt.figure(figsize = (16, 10))
sb.barplot(x = contrib_ind_df.index, 
                y = "Dim1",
                palette = ["lightseagreen"],
                data = contrib_ind_df.sort_values(by = "Dim1", ascending = False))
plt.axhline(y = 100/n, color = 'red', linestyle = '--', linewidth = 1)


In [ ]:
plt.figure(figsize = (16, 10))
sb.barplot(x = contrib_ind_df.index, 
                y = "Dim2",
                palette = ["lightseagreen"],
                data = contrib_ind_df.sort_values(by = "Dim2", ascending = False))
plt.axhline(y = 100/n, color = 'red', linestyle = '--', linewidth = 1)


Qualité de représentation (cos2) individus

In [ ]:
# cos2 individus
cos2_ind_df=np.square(coord_ind_df).div(np.square(np.linalg.norm(coord_ind,axis=1)),axis="rows")
cos2_ind_df.round(3)

Coordonnées des variables actives

In [ ]:
# correction (n-1)/n car variance d'échantillonage utilisée
coord_var = t_pca.components_.T*np.sqrt(t_pca.explained_variance_)*(n-1)/n)
coord_var_df = pd.DataFrame(coord_var[:,:2], 
                            columns = ["Dim" + str(i+1) for i in range(2)], 
                            index = tempn.columns[:12])
coord_var_df.round(3)

Coordonnées variables quantitatives supplémentaires

In [ ]:
var_sup_cr=scale(temp[temp.columns[13:]])

# Idem
coord_var_sup = var_sup_cr.T.dot(coord_ind/np.sqrt(t_pca.explained_variance_*(n-1)/n)/n)
coord_var_sup_df = pd.DataFrame(coord_var_sup[:,:2], 
                            columns = ["Dim" + str(i+1) for i in range(2)], 
                            index = temp.columns[13:])
coord_var_sup_df.round(3)

Contribution variables

In [ ]:
# Idem 
contrib_var_df=np.square(coord_var_df).div(t_pca.explained_variance_[:2]*(n-1)/n, axis='columns')*100
contrib_var_df.round(3)

Qualité de représentation (cos2) variable

In [ ]:
# Ici comme c'est une ACP normée les vecteurs variables sont
# de longueur 1, donc le cos2 est leur coordonnée au carré 
cos2_var_df=np.square(coord_var_df)
cos2_var_df.round(3)                                       

In [ ]:
# cos2 variables sup
cos2_var_sup_df=np.square(coord_var_sup_df)
cos2_var_sup_df.round(3)                                       

Variables plan 1-2

In [ ]:
fig, axes = plt.subplots(figsize = (10, 10))
fig.suptitle("Cercle des corrélations")
axes.set_xlim(-1, 1)
axes.set_ylim(-1, 1)
axes.axvline(x = 0, color = 'lightgray', linestyle = '--', linewidth = 1)
axes.axhline(y = 0, color = 'lightgray', linestyle = '--', linewidth = 1)
axes.set_xlabel("Dim1")
axes.set_ylabel("Dim2")
for j in range(12):
    axes.text(coord_var_df["Dim1"][j],coord_var_df["Dim2"][j], coord_var_df.index[j], size = 15)
    axes.plot([0,coord_var_df["Dim1"][j]], [0,coord_var_df["Dim2"][j]], color = "gray", linestyle = 'dashed')
plt.gca().add_artist(plt.Circle((0,0),1,color='blue',fill=False))
for j in range(4):
    axes.text(coord_var_sup_df["Dim1"][j],coord_var_sup_df["Dim2"][j], coord_var_sup_df.index[j], size = 15)
    axes.plot([0,coord_var_sup_df["Dim1"][j]], [0,coord_var_sup_df["Dim2"][j]], color = "blue", linestyle = 'dashed')

plt.show()

Individus plan 1-2

In [ ]:
plt.figure(figsize = (16, 10))
sb.scatterplot(x = "Dim1", y = "Dim2", data =coord_ind_df , color = "gray", alpha = 0)

for i in range(coord_ind_df.shape[0]):
    C='black'
    if (contrib_ind_df.iloc[i][0] > 100/n):
        C='Orange'
    if (contrib_ind_df.iloc[i][1] > 100/n):
        C = 'Red'
    if ((contrib_ind_df.iloc[i][1] > 100/n) and (contrib_ind_df.iloc[i][0] > 100/n)):
        C = 'Green'
    plt.text(coord_ind_df.iloc[i][0], coord_ind_df.iloc[i][1], coord_ind_df.index[i], 
             ha = "center", va = "center", fontsize=12,  color=C)
plt.axvline(x = 0, color = 'lightgray', linestyle = '--', linewidth = 1)
plt.axhline(y = 0, color = 'lightgray', linestyle = '--', linewidth = 1)
